# 01. Проверка качества данных продаж

## 1. Загрузка данных

In [8]:
import pandas as pd
from pathlib import Path

data_path = Path.cwd().parent.parent / "data" / "clean"
sales_data_path = data_path / "sales_data.csv"
stocks_data_path = data_path / "current_stocks_data.csv"

sales_data = pd.read_csv(sales_data_path)
stocks_data = pd.read_csv(stocks_data_path)

## 2. Общая информация о данных

In [9]:
sale_columns = sales_data.shape[0]
stock_columns = stocks_data.shape[0]
sku_counts = sales_data["sku"].value_counts().describe()
rare_skus = sales_data["sku"].value_counts().loc[lambda x: x <= 3]

print(f"Количество строк в sales_data: {sale_columns}")
print(f"Количество строк в stocks_data: {stock_columns}")
print("Статистика по количеству записей на SKU в sales_data:")
print(sku_counts)
print("Редкие SKU (менее или равно 2 записям):")
print(rare_skus)
print("Количесво редких SKU:", rare_skus.count())

Количество строк в sales_data: 6060
Количество строк в stocks_data: 548
Статистика по количеству записей на SKU в sales_data:
count    725.000000
mean       7.451034
std        3.386990
min        1.000000
25%        5.000000
50%        9.000000
75%       11.000000
max       11.000000
Name: count, dtype: float64
Редкие SKU (менее или равно 2 записям):
sku
КО25985    3
ПЗ17536    3
КО25456    3
ЯП25415    3
ЯП11969    3
          ..
КО17374    1
0104       1
РФ23890    1
ПЗ17841    1
РФ10537    1
Name: count, Length: 132, dtype: int64
Количесво редких SKU: 132


## 3. Оценка полноты временной истории по товарам

In [10]:
sku_counts = sales_data.groupby("sku")["month"].nunique()

history_df = sku_counts.reset_index(name="months_count")

history_df["history_group"] = pd.cut(
    history_df["months_count"],
    bins=[0, 2, 5, 8, 11],
    labels=["1–2 мес", "3–5 мес", "6–8 мес", "9–11 мес"]
)

history_df["history_group"].value_counts()

history_group
9–11 мес    363
6–8 мес     144
3–5 мес     124
1–2 мес      94
Name: count, dtype: int64

Анализ показал, что значительная часть ассортимента имеет неполную историю продаж. Около 218 товаров имеют менее 3 месяцев данных и не могут быть использованы для автоматического прогнозирования. Для дальнейших этапов анализа требуется дополнительная классификация товаров по стабильности спроса.

## 4. Выводы и ограничения
В результате проверки данных установлено, что датасет содержит 725 уникальных SKU с помесячной историей продаж за период январь–ноябрь.
Медианное количество месяцев продаж на товар составляет 9, однако около 25% ассортимента имеют менее 5 месяцев истории, что ограничивает возможность применения автоматического прогнозирования для этих позиций.
Данные в целом корректны и пригодны для дальнейшего анализа. Для следующего этапа требуется классификация товаров по стабильности спроса

## 5. Решение по результатам проверки
Для дальнейшего анализа будут использоваться только товары с достаточной историей продаж.
На следующем этапе планируется провести XYZ-анализ для оценки стабильности спроса и определения стратегии прогнозирования и управления запасами

In [11]:
history_df["history_group"].value_counts(normalize=True) * 100

history_group
9–11 мес    50.068966
6–8 мес     19.862069
3–5 мес     17.103448
1–2 мес     12.965517
Name: proportion, dtype: float64

69,931035% товаров имеют полную историю (9–11 месяцев), 30,068965% — ограниченную.